# 🏦 Unicaja Banking Assistant — RAG + StableLM

Este notebook demuestra cómo construir un asistente bancario inteligente usando:

- 📄 Documentos PDF reales (ej: guía hipotecaria de Unicaja)
- 🔍 Recuperación aumentada (RAG)
- 🧠 Modelo local de lenguaje (`stablelm-tuned-alpha-3b`)
- 🔡 Embeddings semánticos (`all-MiniLM-L6-v2`)

---

## ⚙️ Etapas del Proyecto

### 1️⃣ Carga y segmentación del PDF
Dividimos el documento en partes manejables (*chunks*) para su posterior análisis.

In [1]:
import os
print(os.getcwd())

/Users/luisdotto/Documents/CODE/Projects/Information_Technology/GenerativeAI-Banking-RAG-Chatbot/notebooks


In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

### 1️⃣ Carga y segmentación del PDF
Dividimos el PDF en bloques de texto más pequeños para poder analizarlos y vectorizarlos.

In [3]:
from src.loader import ocr_pdf_to_text_chunks


In [4]:
from src.loader import ocr_pdf_to_text_chunks


chunks = ocr_pdf_to_text_chunks("../data/Spain_unicaja_Fixed_Mortage.pdf")
print(chunks[0])  # muestra el primer chunk


✅ OCR extraído y dividido en 10 chunks
Unicaja Banco, S.A., Avda. Andalucia 10 - 12, Malaga. Inscrito en el Registro Mercantil de Malaga, Tomo 4.952, Libro 3.859, Seccidn 8, Hoja MA-111.580, Folio 1, Inscripcidn 1°. N.I.F. 493139053.

© Unicaja Banco, S.A. Todos los derechos reservados. Prohibida la reproduccion total o parcial por cualquier medio fisico o digital sin autorizacidn expresa de Unicaja Banco, S.A.

| Unicaja

Tu hipoteca 100% online
en 4 sencillos pasos

Simulacion

" Te informamos que tienes disponible en la Web la inf


### 2️⃣ Embeddings y almacenamiento semántico
Convertimos los chunks en vectores numéricos y los almacenamos en una base vectorial Chroma.

In [5]:
from langchain.schema import Document
from src.embeddings import embed_and_store_documents


docs = [Document(page_content=chunk) for chunk in chunks]
db = embed_and_store_documents(docs)

/Users/luisdotto/Documents/CODE/Projects/Information_Technology/GenerativeAI-Banking-RAG-Chatbot/Lang/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/luisdotto/Documents/CODE/Projects/Information_Technology/GenerativeAI-Banking-RAG-Chatbot/src/embeddings.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name=model_name)
/Users/luisdotto/Documents/CODE/Projects/Information_Technology/GenerativeAI-Banking-RAG

✅ Chroma DB created and saved in 'db/chroma/'


/Users/luisdotto/Documents/CODE/Projects/Information_Technology/GenerativeAI-Banking-RAG-Chatbot/src/embeddings.py:29: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


### 3️⃣ Preparación del modelo LLM
Cargamos el modelo `StableLM` y lo conectamos con LangChain para que genere respuestas.

In [6]:
from src.rag_chain import build_rag_chain

qa = build_rag_chain(model_id="google/flan-t5-base") 
response = qa.invoke({"query": "¿Qué condiciones tiene la hipoteca fija?"})
print(response["result"])


/Users/luisdotto/Documents/CODE/Projects/Information_Technology/GenerativeAI-Banking-RAG-Chatbot/src/rag_chain.py:27: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(
Device set to use mps
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 

✅ RAG chain with custom prompt ready
Responde a la siguiente pregunta usando únicamente el contexto proporcionado. 
Si no encontrás la información, respondé con: "No tengo esa info, lo siento."

Sé claro, informal pero educado. Redactá la respuesta como si fueras un asistente bancario que quiere ayudar a una persona real.

Contexto:
tu hipoteca (forma de pago, plazo, cuota...) y te pedimos que
aportes la documentacion necesaria para el estudio de la solicitud.

Necesitaremos, entre otros, estos documentos en formato digital por cada titular:

« Documento de identidad.

- Informe de Vida Laboral (Seguridad Social).

- Si tienes ya otros préstamos, los 3 ultimos recibos de cada uno de ellos.
- Ultima declaracion de la Renta (IRPF).

* 3ultimas nominas.

« Nota Simple de la vivienda.

Ademas, el importe ofrecido no podra super

tu hipoteca (forma de pago, plazo, cuota...) y te pedimos que
aportes la documentacion necesaria para el estudio de la solicitud.

Necesitaremos, entre otros, esto

### 4️⃣ Consulta al sistema RAG
Probamos el asistente con una pregunta real en lenguaje natural.


pregunta = "¿Cuáles son los pasos para contratar una hipoteca?"
respuesta = rag.run(pregunta)
print("🤖", respuesta)